# Dynamic Prompt
<img src="./assets/LC_DynamicPrompts.png" width="500">

## Setup

Load and/or check for needed environmental variables

In [ ]:
%%capture
%pip install --upgrade langgraph langchain langchain-ollama langchain-community
%pip install --upgrade python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Ollama server URL - override with OLLAMA_BASE_URL env var
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "https://ungambolling-dwain-unsprouted.ngrok-free.dev")

In [ ]:
# (Database logic removed in this version.)
# This notebook now demonstrates a dynamic prompt for an email translator.

In [ ]:
# No database tools are needed for the email translator example.
# The dynamic prompt will only rely on `RuntimeContext` values.

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """You are an assistant that translates and rewrites emails.

Target language: {target_language}
Writing style: {style_instructions}

Guidelines:
- Translate the user's email into the target language.
- Use the requested writing style.
- Preserve the original meaning, dates, names, and numbers.
- Improve clarity and professionalism when appropriate.
"""

## Build a Dynamic Prompt
Utilize runtime context and middleware to generate a dynamic prompt.

In [ ]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    context = request.runtime.context

    if context.is_formal:
        style_instructions = (
            "a formal, professional, and polite business tone"
        )
    else:
        style_instructions = (
            "a casual, friendly, and conversational tone"
        )

    return SYSTEM_PROMPT_TEMPLATE.format(
        target_language=context.target_language,
        style_instructions=style_instructions,
    )

Include middleware in `create_agent`.

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

chat = ChatOllama(
    model="llama3.1",
    temperature=0.7,
    base_url=OLLAMA_BASE_URL,
)

agent = create_agent(
    model=chat,
    tools=[],  # No tools needed; the model only translates emails.
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [ ]:
question = (
    "Please translate the following email and make it sound clear "
    "and natural:\n\n"
    "Hi team, I'm running a bit late today but will join the "
    "meeting as soon as I can. Thanks for your understanding!"
 )

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_formal=False, target_language="French"),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
question = (
    "Please translate the following email into a very professional "
    "version:\n\n"
    "Hi team, I'm running a bit late today but will join the "
    "meeting as soon as I can. Thanks for your understanding!"
 )

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_formal=True, target_language="French"),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()